In [1]:
import pandas as pd

file_path = "train.csv"

# Read only 5 rows first
df_sample = pd.read_csv(file_path, nrows=5)

print("Shape:", df_sample.shape)

print("\nColumns:")
print(df_sample.columns.tolist())

Shape: (5, 12)

Columns:
['channelGrouping', 'date', 'device', 'fullVisitorId', 'geoNetwork', 'sessionId', 'socialEngagementType', 'totals', 'trafficSource', 'visitId', 'visitNumber', 'visitStartTime']


In [2]:
for i, col in enumerate(df_sample.columns, start=1):
    print(i, ":", col)

1 : channelGrouping
2 : date
3 : device
4 : fullVisitorId
5 : geoNetwork
6 : sessionId
7 : socialEngagementType
8 : totals
9 : trafficSource
10 : visitId
11 : visitNumber
12 : visitStartTime


In [3]:
import json

# Display one row in a readable format
for col in ["device", "geoNetwork", "totals", "trafficSource"]:
    print("\n" + "=" * 60)
    print(col)
    print("=" * 60)
    print(df_sample.iloc[0][col])


device
{"browser": "Chrome", "browserVersion": "not available in demo dataset", "browserSize": "not available in demo dataset", "operatingSystem": "Windows", "operatingSystemVersion": "not available in demo dataset", "isMobile": false, "mobileDeviceBranding": "not available in demo dataset", "mobileDeviceModel": "not available in demo dataset", "mobileInputSelector": "not available in demo dataset", "mobileDeviceInfo": "not available in demo dataset", "mobileDeviceMarketingName": "not available in demo dataset", "flashVersion": "not available in demo dataset", "language": "not available in demo dataset", "screenColors": "not available in demo dataset", "screenResolution": "not available in demo dataset", "deviceCategory": "desktop"}

geoNetwork
{"continent": "Asia", "subContinent": "Western Asia", "country": "Turkey", "region": "Izmir", "metro": "(not set)", "city": "Izmir", "cityId": "not available in demo dataset", "networkDomain": "ttnet.com.tr", "latitude": "not available in demo 

In [4]:
import pandas as pd
import json
from pathlib import Path

input_file = "train.csv"
output_file = "ga_funnel_analysis.csv"

# Process the large CSV in chunks
chunksize = 10000
first_chunk = True

for chunk in pd.read_csv(input_file, chunksize=chunksize):

    # Extract JSON columns
    def get_value(x, key):
        try:
            if pd.isna(x):
                return None
            return json.loads(x).get(key)
        except:
            return None

    chunk["device_category"] = chunk["device"].apply(
        lambda x: get_value(x, "deviceCategory")
    )

    chunk["operating_system"] = chunk["device"].apply(
        lambda x: get_value(x, "operatingSystem")
    )

    chunk["browser"] = chunk["device"].apply(
        lambda x: get_value(x, "browser")
    )

    chunk["country"] = chunk["geoNetwork"].apply(
        lambda x: get_value(x, "country")
    )

    chunk["source"] = chunk["trafficSource"].apply(
        lambda x: get_value(x, "source")
    )

    chunk["medium"] = chunk["trafficSource"].apply(
        lambda x: get_value(x, "medium")
    )

    chunk["campaign"] = chunk["trafficSource"].apply(
        lambda x: get_value(x, "campaign")
    )

    chunk["sessions"] = chunk["totals"].apply(
        lambda x: get_value(x, "visits")
    )

    chunk["pageviews"] = chunk["totals"].apply(
        lambda x: get_value(x, "pageviews")
    )

    chunk["bounces"] = chunk["totals"].apply(
        lambda x: get_value(x, "bounces")
    )

    chunk["new_visits"] = chunk["totals"].apply(
        lambda x: get_value(x, "newVisits")
    )

    chunk["revenue"] = chunk["totals"].apply(
        lambda x: get_value(x, "transactionRevenue")
    )

    # Select only the fields needed for our project
    result = chunk[
        [
            "date",
            "channelGrouping",
            "fullVisitorId",
            "sessionId",
            "device_category",
            "operating_system",
            "browser",
            "country",
            "source",
            "medium",
            "campaign",
            "sessions",
            "pageviews",
            "bounces",
            "new_visits",
            "revenue"
        ]
    ]

    # Convert numeric columns
    for col in ["sessions", "pageviews", "bounces",
                "new_visits", "revenue"]:
        result[col] = pd.to_numeric(result[col], errors="coerce").fillna(0)

    # Create conversion flag
    result["conversion"] = (result["revenue"] > 0).astype(int)

    # Save chunk
    result.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

print("Processing completed!")
print("Created:", output_file)

Processing completed!
Created: ga_funnel_analysis.csv


In [5]:
import pandas as pd

df = pd.read_csv("ga_funnel_analysis.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

C:\Users\HP\AppData\Local\Temp\ipykernel_8756\1107463953.py:3: DtypeWarning: Columns (0: fullVisitorId) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("ga_funnel_analysis.csv")


Rows: 903653
Columns: 17

Columns:
['date', 'channelGrouping', 'fullVisitorId', 'sessionId', 'device_category', 'operating_system', 'browser', 'country', 'source', 'medium', 'campaign', 'sessions', 'pageviews', 'bounces', 'new_visits', 'revenue', 'conversion']

First 5 rows:


,date,channelGrouping,fullVisitorId,sessionId,device_category,operating_system,browser,country,source,medium,campaign,sessions,pageviews,bounces,new_visits,revenue,conversion
0,20160902,Organic Search,1131660440785968503,1131660440785968503_1472830385,desktop,Windows,Chrome,Turkey,google,organic,(not set),1,1.0,1.0,1.0,0.0,0
1,20160902,Organic Search,377306020877927890,377306020877927890_1472880147,desktop,Macintosh,Firefox,Australia,google,organic,(not set),1,1.0,1.0,1.0,0.0,0
2,20160902,Organic Search,3895546263509774583,3895546263509774583_1472865386,desktop,Windows,Chrome,Spain,google,organic,(not set),1,1.0,1.0,1.0,0.0,0
3,20160902,Organic Search,4763447161404445595,4763447161404445595_1472881213,desktop,Linux,UC Browser,Indonesia,google,organic,(not set),1,1.0,1.0,1.0,0.0,0
4,20160902,Organic Search,27294437909732085,27294437909732085_1472822600,mobile,Android,Chrome,United Kingdom,google,organic,(not set),1,1.0,1.0,0.0,0.0,0


In [6]:
print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)


Missing values:
date                0
channelGrouping     0
fullVisitorId       0
sessionId           0
device_category     0
operating_system    0
browser             0
country             0
source              0
medium              0
campaign            0
sessions            0
pageviews           0
bounces             0
new_visits          0
revenue             0
conversion          0
dtype: int64

Data types:
date                  int64
channelGrouping         str
fullVisitorId        object
sessionId               str
device_category         str
operating_system        str
browser                 str
country                 str
source                  str
medium                  str
campaign                str
sessions              int64
pageviews           float64
bounces             float64
new_visits          float64
revenue             float64
conversion            int64
dtype: object


In [9]:
# Make a copy for analysis
analysis = df.copy()

# Convert revenue from microunits to normal currency
analysis["revenue"] = analysis["revenue"] / 1_000_000

# Engagement definition
analysis["engaged_session"] = (
    analysis["pageviews"] > 1
).astype(int)

# Basic KPIs
total_visitors = analysis["fullVisitorId"].nunique()

total_sessions = len(analysis)

engaged_sessions = analysis["engaged_session"].sum()

conversions = analysis["conversion"].sum()

total_revenue = analysis["revenue"].sum()

# Conversion rates
engagement_rate = (
    engaged_sessions / total_sessions * 100
)

conversion_rate = (
    conversions / total_sessions * 100
)

engaged_to_conversion = (
    conversions / engaged_sessions * 100
    if engaged_sessions > 0 else 0
)

revenue_per_visitor = (
    total_revenue / total_visitors
)

# Display results
print("========== MARKETING FUNNEL ==========")

print(f"Total Visitors       : {total_visitors:,}")
print(f"Total Sessions       : {total_sessions:,}")
print(f"Engaged Sessions     : {engaged_sessions:,}")
print(f"Conversions          : {conversions:,}")
print(f"Total Revenue        : {total_revenue:,.2f}")

print("\n========== CONVERSION METRICS ==========")

print(f"Engagement Rate      : {engagement_rate:.2f}%")
print(f"Session Conversion   : {conversion_rate:.2f}%")
print(f"Engaged Conversion   : {engaged_to_conversion:.2f}%")
print(f"Revenue / Visitor    : {revenue_per_visitor:.2f}")

========== MARKETING FUNNEL ==========
Total Visitors       : 714,167
Total Sessions       : 903,653
Engaged Sessions     : 451,031
Conversions          : 11,515
Total Revenue        : 1,540,071.24

========== CONVERSION METRICS ==========
Engagement Rate      : 49.91%
Session Conversion   : 1.27%
Engaged Conversion   : 2.55%
Revenue / Visitor    : 2.16


In [10]:
# Channel performance analysis

channel_analysis = (
    analysis
    .groupby("channelGrouping")
    .agg(
        Visitors=("fullVisitorId", "nunique"),
        Sessions=("sessionId", "count"),
        Engaged_Sessions=("engaged_session", "sum"),
        Conversions=("conversion", "sum"),
        Revenue=("revenue", "sum")
    )
    .reset_index()
)

# Calculate metrics
channel_analysis["Engagement_Rate_%"] = (
    channel_analysis["Engaged_Sessions"]
    / channel_analysis["Sessions"] * 100
)

channel_analysis["Conversion_Rate_%"] = (
    channel_analysis["Conversions"]
    / channel_analysis["Sessions"] * 100
)

channel_analysis["Revenue_per_Visitor"] = (
    channel_analysis["Revenue"]
    / channel_analysis["Visitors"]
)

# Sort by conversion rate
channel_analysis = channel_analysis.sort_values(
    "Conversion_Rate_%",
    ascending=False
)

display(channel_analysis)

,channelGrouping,Visitors,Sessions,Engaged_Sessions,Conversions,Revenue,Engagement_Rate_%,Conversion_Rate_%,Revenue_per_Visitor
6,Referral,65661,104838,77320,5311,651429.91,73.751884,5.065911,9.921109
3,Display,4103,6262,3999,142,78337.46,63.861386,2.267646,19.092727
5,Paid Search,18702,25326,15666,468,43558.90,61.857380,1.847903,2.329104
2,Direct,109830,143026,71658,2042,434840.55,50.101380,1.427712,3.959215
4,Organic Search,311607,381561,196623,3438,326380.51,51.531210,0.901035,1.047411
0,(Other),90,120,64,1,9.99,53.333333,0.833333,0.111000
1,Affiliates,13400,16403,7643,9,597.38,46.595135,0.054868,0.044581
7,Social,212374,226117,78058,104,4916.54,34.521067,0.045994,0.023150


In [11]:
channel_analysis.to_csv(
    "channel_performance.csv",
    index=False
)

print("Channel analysis saved successfully!")

Channel analysis saved successfully!


In [12]:
# Display complete channel analysis
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print(channel_analysis.to_string(index=False))

channelGrouping  Visitors  Sessions  Engaged_Sessions  Conversions   Revenue  Engagement_Rate_%  Conversion_Rate_%  Revenue_per_Visitor
       Referral     65661    104838             77320         5311 651429.91          73.751884           5.065911             9.921109
        Display      4103      6262              3999          142  78337.46          63.861386           2.267646            19.092727
    Paid Search     18702     25326             15666          468  43558.90          61.857380           1.847903             2.329104
         Direct    109830    143026             71658         2042 434840.55          50.101380           1.427712             3.959215
 Organic Search    311607    381561            196623         3438 326380.51          51.531210           0.901035             1.047411
        (Other)        90       120                64            1      9.99          53.333333           0.833333             0.111000
     Affiliates     13400     16403             

In [13]:
channel_analysis.to_csv(
    "channel_performance.csv",
    index=False
)

print("✅ Channel performance file ready for Power BI")

✅ Channel performance file ready for Power BI


In [14]:
# Make sure date is in proper format
analysis["date"] = pd.to_datetime(
    analysis["date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

# Create month field
analysis["month"] = analysis["date"].dt.to_period("M").astype(str)

# Monthly performance
monthly_analysis = (
    analysis
    .groupby("month")
    .agg(
        Visitors=("fullVisitorId", "nunique"),
        Sessions=("sessionId", "count"),
        Engaged_Sessions=("engaged_session", "sum"),
        Conversions=("conversion", "sum"),
        Revenue=("revenue", "sum")
    )
    .reset_index()
)

# Calculate rates
monthly_analysis["Engagement_Rate_%"] = (
    monthly_analysis["Engaged_Sessions"]
    / monthly_analysis["Sessions"] * 100
)

monthly_analysis["Conversion_Rate_%"] = (
    monthly_analysis["Conversions"]
    / monthly_analysis["Sessions"] * 100
)

monthly_analysis["Revenue_per_Visitor"] = (
    monthly_analysis["Revenue"]
    / monthly_analysis["Visitors"]
)

# Display
display(monthly_analysis)

,month,Visitors,Sessions,Engaged_Sessions,Conversions,Revenue,Engagement_Rate_%,Conversion_Rate_%,Revenue_per_Visitor
0,2016-08,61699,74759,37917,1119,154666.56,50.718977,1.496810,2.506792
1,2016-09,59121,71032,35958,859,126031.24,50.622255,1.209314,2.131751
2,2016-10,84901,97506,44594,872,113329.07,45.734621,0.894304,1.334838
3,2016-11,99734,113972,56276,919,119013.87,49.377040,0.806338,1.193313
4,2016-12,63839,79124,43497,1395,154557.93,54.973207,1.763055,2.421058
5,2017-01,53041,64694,33276,697,97877.79,51.435991,1.077380,1.845323
6,2017-02,51364,62192,30762,708,108756.52,49.462953,1.138410,2.117369
7,2017-03,57888,69931,33923,883,130964.27,48.509245,1.262673,2.262373
8,2017-04,55681,67126,32487,928,158788.80,48.397044,1.382475,2.851759
9,2017-05,52233,65371,34040,1115,121711.48,52.072020,1.705649,2.330164


In [15]:
monthly_analysis.to_csv(
    "monthly_performance.csv",
    index=False
)

print("✅ Monthly performance file ready for Power BI")

✅ Monthly performance file ready for Power BI


In [16]:
device_analysis = (
    analysis
    .groupby("device_category")
    .agg(
        Visitors=("fullVisitorId", "nunique"),
        Sessions=("sessionId", "count"),
        Engaged_Sessions=("engaged_session", "sum"),
        Conversions=("conversion", "sum"),
        Revenue=("revenue", "sum")
    )
    .reset_index()
)

# Calculate performance metrics
device_analysis["Engagement_Rate_%"] = (
    device_analysis["Engaged_Sessions"]
    / device_analysis["Sessions"] * 100
)

device_analysis["Conversion_Rate_%"] = (
    device_analysis["Conversions"]
    / device_analysis["Sessions"] * 100
)

device_analysis["Revenue_per_Visitor"] = (
    device_analysis["Revenue"]
    / device_analysis["Visitors"]
)

# Sort by revenue
device_analysis = device_analysis.sort_values(
    "Revenue",
    ascending=False
)

display(device_analysis)

,device_category,Visitors,Sessions,Engaged_Sessions,Conversions,Revenue,Engagement_Rate_%,Conversion_Rate_%,Revenue_per_Visitor
0,desktop,523690,664479,335927,10495,1480864.09,50.554946,1.579433,2.827749
1,mobile,166173,208725,99850,852,49785.81,47.838064,0.408193,0.299602
2,tablet,24304,30449,15254,168,9421.34,50.096883,0.551742,0.387646


In [17]:
device_analysis.to_csv(
    "device_performance.csv",
    index=False
)

print("✅ Device performance file ready for Power BI")

✅ Device performance file ready for Power BI


In [18]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Visitors",
        "Total Sessions",
        "Engaged Sessions",
        "Conversions",
        "Total Revenue",
        "Engagement Rate",
        "Conversion Rate",
        "Engaged Conversion Rate",
        "Revenue per Visitor"
    ],
    "Value": [
        total_visitors,
        total_sessions,
        engaged_sessions,
        conversions,
        total_revenue,
        engagement_rate,
        conversion_rate,
        engaged_to_conversion,
        revenue_per_visitor
    ]
})

kpi_summary.to_csv("kpi_summary.csv", index=False)

print("✅ KPI summary created")
display(kpi_summary)

✅ KPI summary created


,KPI,Value
0,Total Visitors,7.141670e+05
1,Total Sessions,9.036530e+05
2,Engaged Sessions,4.510310e+05
3,Conversions,1.151500e+04
4,Total Revenue,1.540071e+06
5,Engagement Rate,4.991197e+01
6,Conversion Rate,1.274272e+00
7,Engaged Conversion Rate,2.553040e+00
8,Revenue per Visitor,2.156458e+00


In [19]:
kpi_summary_clean = pd.DataFrame({
    "Total Visitors": [total_visitors],
    "Total Sessions": [total_sessions],
    "Engaged Sessions": [engaged_sessions],
    "Conversions": [conversions],
    "Total Revenue": [total_revenue],
    "Engagement Rate %": [engagement_rate],
    "Conversion Rate %": [conversion_rate],
    "Engaged Conversion Rate %": [engaged_to_conversion],
    "Revenue per Visitor": [revenue_per_visitor]
})

kpi_summary_clean.to_csv(
    "kpi_summary_clean.csv",
    index=False
)

display(kpi_summary_clean)

,Total Visitors,Total Sessions,Engaged Sessions,Conversions,Total Revenue,Engagement Rate %,Conversion Rate %,Engaged Conversion Rate %,Revenue per Visitor
0,714167,903653,451031,11515,1540071.24,49.911968,1.274272,2.55304,2.156458
